In [2]:
import sys
import os
sys.path.append(os.path.abspath('c:/Users/loren/Documents/Tortues/github/Vnet_seaturtles_semisupervised/src'))

In [3]:
from Vnet_architecture_pytorch import *
from Trainer_pytorch import *
from Global_generator_pytorch import *
from Data import *
from Prediction_pytorch import *
from Prediction import *

In [4]:
import torch
from pathlib import Path
from os import listdir
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import datetime
import gc
import time

In [5]:
class classCLASS:

    def __init__(self,id,name,behaviors,score_weight,sampling_weight=1,loss_weight=1):
        self.id=id
        self.name=name
        self.behaviors=behaviors
        self.score_weight=score_weight
        self.sampling_weight=sampling_weight
        self.loss_weight=loss_weight
        

In [16]:
DIR_MODEL="C:/Users/loren/Documents/Tortues/Martinique_Verte/Model_results/model=2026_08_04_TestA_pressurecorrected/"
DIR_IN="D:/Martinique/Leo/TV_processed_1/"



In [17]:
CASE=DIR_MODEL[-33:-1]
CASE


'26_08_04_TestA_pressurecorrected'

In [18]:
NAMES_TV=[f[0:-4] for f in listdir(DIR_IN) if f.endswith(".npy")]

In [19]:

now = datetime.datetime.now()
today=now.strftime("%Y_%m_%d")
today

DIR_OUT="C:/Users/loren/Documents/Tortues/Martinique_Verte/Predictions="+today+"_"+CASE



import shutil
if os.path.exists(DIR_OUT):
    #shutil.rmtree(DIR_OUT)
    print("the directory already created:",DIR_OUT)
else:
    print("we create the directory:",DIR_OUT)
    os.makedirs(DIR_OUT)


the directory already created: C:/Users/loren/Documents/Tortues/Martinique_Verte/Predictions=2026_08_13_26_08_04_TestA_pressurecorrected


In [20]:
Other=classCLASS(0,"Other",None,0,1,0.1) 
Breathing=classCLASS(1,"Breathing",[1,2,40],1)
Feeding=classCLASS(2,"Feeding",[3,4,5,6,7,11,12,13,18,19,20,21],10,2,2)
Gliding=classCLASS(3,"Gliding",[16,17],1,3) 
Resting=classCLASS(4,"Resting",[29,30,31,32],1,0.5)
Scratching=classCLASS(5,"Scratching",[34,35,36],2,2,2)
Swimming=classCLASS(6,"Swimming",[42,45,48,43,46,49,27,44,47,50,56,52,54,55],1,0.5) 
CLASSES=[Other,Breathing,Feeding,Gliding,Resting,Scratching,Swimming]

In [21]:
DESCRIPTORS_SELEC=[0,1,2,3,4,5,9] 
WINDOW_DURA=40 

RESAMPLING_STEP_S=0.0500
FREQ=1/RESAMPLING_STEP_S
WINDOW_SIZE=int(WINDOW_DURA/RESAMPLING_STEP_S)
BATCH_SIZE=32
MODEL_DEPTH=32
DROPOUT=0.5

## Load the dataset

In [22]:
data=Data(DIR_IN, DIR_MODEL,  DESCRIPTORS_SELEC)
dico=data.load_dico()
data.print_selected_variables(dico)


 0 Accelerometer X [m/s²]
 1 Accelerometer Y [m/s²]
 2 Accelerometer Z [m/s²]
 3 Gyroscope X [mrad/s]
 4 Gyroscope Y [mrad/s]
 5 Gyroscope Z [mrad/s]
-----suppressed-----> 6 Depth
-----suppressed-----> 7 Light intensity 1 [raw]
-----suppressed-----> 8 Temperature (depth) [°C]
 9 Pressure_diff


In [23]:
matrices=data.load_data_guyane(NAMES_TV, FREQ)

In [24]:
matrices=data.normalize(matrices)

## Load the model

In [29]:
#the Behavioral categories defined by classCLASS : id ,name of behavior ,behaviors number in the data csv, sampling_weight=1,loss_weight=1
Other=classCLASS(0,"Other",None,1)
Breathing=classCLASS(1,"Breathing",[1,2,40],1)
Feeding=classCLASS(2,"Feeding",[3,4,5,6,7,11,12,13,18,19,20,21],2,2)
Gliding=classCLASS(3,"Gliding",[16,17],3)
Resting=classCLASS(4,"Resting",[29,30,31,32],0.5)
Scratching=classCLASS(5,"Scratching",[34,35,36],2,2)
Swimming=classCLASS(6,"Swimming",[42,45,48,43,46,49,27,44,47,50,56,52,54,55],0.5)

CLASSES=[Other,Breathing,Feeding,Gliding,Resting,Scratching,Swimming]
CLASS_NAMES=[CLASS.name for CLASS in CLASSES]

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model= VNet4Levels(len(DESCRIPTORS_SELEC), len(CLASSES), MODEL_DEPTH, dropout_rate=DROPOUT)
model.load_state_dict(torch.load(os.path.join(Path(DIR_MODEL, "model.pth")), weights_only=True))
model.eval()
model.to(device)

VNet4Levels(
  (dropout): SpatialDropout1D(
    (dropout): Dropout2d(p=0.5, inplace=False)
  )
  (conv1): Sequential(
    (0): Conv1d(7, 32, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
    (4): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
  )
  (down1): Conv1d(32, 64, kernel_size=(2,), stride=(2,))
  (conv2): Sequential(
    (0): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
    (4): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
  )
  (down2): Conv1d(64, 128, 

In [30]:
predictor=Prediction_Guyane(NAMES_TV, model, WINDOW_SIZE, matrices, DIR_MODEL, CLASS_NAMES, device, BATCH_SIZE, saving=True )

In [31]:
all_pred=predictor.get_all()

20211024-141045-CC-09-29_TV_102021 (1030601, 7)
20211101-081704-CC-09-28_TV_102021 (964541, 7)
20211102-115134-CC-09-29_TV_102021 (684361, 7)
20211107-072758-CC-09-29_TV_102021 (339741, 7)
20211108-055826-CC-09-28_TV_102021 (1051821, 7)
20211113-213025-CC-09-29_TV_112021 (392621, 7)
20211114-120603-CC-09-28_TV_102021 (1136601, 7)
20211121-203649-CC-09-29_TV_112021 (1108301, 7)
20211122-104139-CC-09-28_TV_102021 (1123741, 7)
20211130-203207-CC-09-29_TV_112021 (1342901, 7)
20220114-125839-CC-09-29_TV_012022 (1032781, 7)
20220414-002025-CC-09-80 (1094901, 7)
20220421-211441-CC-09-80 (1014721, 7)
20220430-135054-CC-09-72 (867701, 7)
20220508-195728-CC-09-80 (239341, 7)
